# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [ ]:
from pathlib import Path

from iacs.audit_system import (
    AuditRunner,
    RequirementCoverageAudit,
    TraceabilityAudit,
    TodoAudit,
)
from iacs.io_system import IOSystem
from iacs.registry import Registry

## Load components.yaml

In [ ]:
io = IOSystem()
components_df = io.read_entity_centered_file(Path("../components/components.yaml"))
registry = Registry.from_entity_centered(components_df)

print(f"Loaded {len(components_df)} components from {components_df['entity_id'].nunique()} entities")
print(f"Component types: {registry.component_types}")

Loaded 280 components from 140 entities
Component types: ['description', 'file_info', 'id', 'input', 'output', 'parent', 'requirement', 'schema', 'system', 'todo']


## Run All Audits

In [ ]:
runner = AuditRunner(audits=[
    RequirementCoverageAudit(),
    TraceabilityAudit(),
    TodoAudit(),
])

results = runner.run(registry)

print(f"All audits passed: {runner.all_passed}")
print()
for name, result in results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"{name}: {status}")

All audits passed: False

requirement_coverage: FAIL
traceability: FAIL
todo: FAIL


## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [ ]:
req_result = results["requirement_coverage"]
print(f"Passed: {req_result.passed}")
print(f"Uncovered requirements: {len(req_result.results)}")
print()

# View uncovered requirements with their descriptions using multi-component join
if not req_result.results.empty:
    uncovered_ids = req_result.results["entity_id"].tolist()
    
    # Get requirements joined with descriptions
    req_with_desc = registry.view(["requirement", "description"])
    
    # Filter to only uncovered requirements
    uncovered_details = req_with_desc[req_with_desc.index.isin(uncovered_ids)]
    
    # Show relevant columns
    display_cols = ["description.value"]
    if "requirement.value" in uncovered_details.columns:
        display_cols.append("requirement.value")
    
    print("Uncovered requirements with descriptions:")
    display(uncovered_details[display_cols])

## Traceability Audit

Checks that all entities trace back to requirements.

In [ ]:
trace_result = results["traceability"]
print(f"Passed: {trace_result.passed}")
print(f"Orphan entities: {len(trace_result.results)}")
print()
if not trace_result.results.empty:
    print("Entities not tracing to requirements:")
    display(trace_result.results)

Passed: False
Orphan entities: 62

Entities not tracing to requirements:


,entity_id
0,file_info
26,iacs.iacs_meta_discussion
27,ecs_framework
35,ecs_framework.requirements.describe_systems.de...
43,ecs_framework.core_concepts
...,...
120,iacs_component.solution
121,iacs_component.task
122,iacs_component.todo
124,data_structure


## Todo Audit

Reports outstanding todos.

In [ ]:
todo_result = results["todo"]
print(f"Passed: {todo_result.passed}")
print(f"Entities with todos: {len(todo_result.results)}")
print()
if todo_result.messages:
    print("Outstanding todos:")
    display(todo_result.results)

Passed: False
Entities with todos: 7

Outstanding todos:


,entity_id
0,iacs.iacs_meta_discussion.syntax_challenges
1,ecs_framework.ecs_system.io_system.to_entity_c...
2,ecs_framework.ecs_system.io_system.write_entit...
3,ecs_framework.ecs_system.io_system.write_entit...
4,ecs_framework.ecs_system.transform_system
5,data_type.str.categorical
6,data_structure.field
